# ⚛️ Kuantum Kapıları### Dr. Buket Toptaş | Kuantum Makine Öğrenmesi Dersi> **Kapsam:** Tek-qubit kapıları (X, Y, Z, H, S, T, Rx, Ry, Rz), çok-qubit kapıları (CNOT, CZ, SWAP, Toffoli), kapı matrislerinin görselleştirilmesi---

## 1. Kuantum Kapısı Nedir?Kuantum kapısı = Qubit'e uygulanan **dönüşüm**. Klasik bilgisayardaki AND, OR, NOT kapılarının kuantum karşılığı.**Temel fark:** Kuantum kapıları her zaman **terslenebilir** (unitary). Girişten çıkışı, çıkıştan girişi her zaman bulabilirsin. Bilgi kaybolmaz.Matematiksel olarak: Kapı = **Üniter matris** (U†U = I)

## 2. Pauli Kapıları (X, Y, Z)### Pauli-X (NOT kapısı)∣0⟩ ↔ ∣1⟩ çevirir. Bloch küresinde **X ekseni** etrafında 180° döndürme.$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad X|0\rangle = |1\rangle, \quad X|1\rangle = |0\rangle$$### Pauli-YX ve Z'nin birleşimi. **Y ekseni** etrafında 180° döndürme.$$Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$$### Pauli-Z (Faz çevirme)∣1⟩'in fazını çevirir. **Z ekseni** etrafında 180° döndürme.$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}, \quad Z|0\rangle = |0\rangle, \quad Z|1\rangle = -|1\rangle$$

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Pauli kapıları — matris ve etkileriX = np.array([[0,1],[1,0]])Y = np.array([[0,-1j],[1j,0]])Z = np.array([[1,0],[0,-1]])I = np.eye(2)ket0 = np.array([[1],[0]])ket1 = np.array([[0],[1]])plus = (ket0 + ket1) / np.sqrt(2)gates = {'X (NOT)': X, 'Y': Y, 'Z (Faz)': Z, 'I (Birim)': I}inputs = {'|0⟩': ket0, '|1⟩': ket1, '|+⟩': plus}print("══════ Pauli Kapılarının Etkileri ══════\n")for gname, gate in gates.items():    print(f"--- {gname} ---")    for iname, inp in inputs.items():        out = gate @ inp        # Format output        a, b = out[0,0], out[1,0]        print(f"  {gname} × {iname} = ({a:.2f})|0⟩ + ({b:.2f})|1⟩")    print()

## 3. Hadamard Kapısı (H)Kuantumun en önemli kapısı! **Süperpozisyon** oluşturur.$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = |+\rangle, \quadH|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |-\rangle$$> **Önemli:** H kapısı kendi tersidir: HH = I. İki kez uygularsan başa dönersin!

In [ ]:
import pennylane as qmlimport numpy as npimport matplotlib.pyplot as pltdev = qml.device('default.qubit', wires=1, shots=1000)@qml.qnode(dev)def hadamard_test(apply_h):    if apply_h >= 1:        qml.Hadamard(wires=0)    if apply_h >= 2:        qml.Hadamard(wires=0)  # İkinci H → başa dön    return qml.counts()fig, axes = plt.subplots(1, 3, figsize=(14, 4))titles = ['|0⟩ (H yok)', 'H|0⟩ = |+⟩', 'HH|0⟩ = |0⟩']for i, ax in enumerate(axes):    result = hadamard_test(i)    states_list = ['0', '1']    counts_list = [result.get(s, 0) for s in states_list]    colors = ['#3b82f6', '#ef4444']    bars = ax.bar(states_list, counts_list, color=colors, alpha=0.85, width=0.5, edgecolor='white')    ax.set_title(titles[i], fontsize=13, fontweight='bold')    ax.set_ylim(0, 1100)    ax.set_ylabel('Sayı' if i==0 else '')    for bar, c in zip(bars, counts_list):        if c > 0:            ax.text(bar.get_x()+bar.get_width()/2, c+20, str(c), ha='center', fontsize=11, fontweight='bold')plt.suptitle('Hadamard Kapısı — Süperpozisyon Oluşturur ve Geri Alır', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

## 4. Rotasyon Kapıları (Rx, Ry, Rz)Bloch küresinde belirli bir açıyla döndürme. **QML'nin temel yapı taşları!**$$R_x(\theta) = \begin{pmatrix} \cos(\theta/2) & -i\sin(\theta/2) \\ -i\sin(\theta/2) & \cos(\theta/2) \end{pmatrix}$$$$R_y(\theta) = \begin{pmatrix} \cos(\theta/2) & -\sin(\theta/2) \\ \sin(\theta/2) & \cos(\theta/2) \end{pmatrix}$$$$R_z(\theta) = \begin{pmatrix} e^{-i\theta/2} & 0 \\ 0 & e^{i\theta/2} \end{pmatrix}$$> **QML bağlantısı:** Variasyonel devrelerde `Ry(θ)` kapıları kullanılır. θ parametresi eğitim sırasında optimize edilir — tıpkı sinir ağındaki ağırlıklar gibi!

In [ ]:
import pennylane as qmlimport numpy as npimport matplotlib.pyplot as pltdev = qml.device('default.qubit', wires=1)# Ry kapısını farklı açılarla deneangles = np.linspace(0, 2*np.pi, 50)prob_0_list = []for theta in angles:    @qml.qnode(dev)    def ry_circuit(t):        qml.RY(t, wires=0)        return qml.probs(wires=0)        probs = ry_circuit(theta)    prob_0_list.append(probs[0])fig, ax = plt.subplots(figsize=(10, 5))ax.plot(np.degrees(angles), prob_0_list, color='#3b82f6', linewidth=2.5, label='P(|0⟩)')ax.plot(np.degrees(angles), [1-p for p in prob_0_list], color='#ef4444', linewidth=2.5, label='P(|1⟩)')ax.axvline(x=90, color='#10b981', linestyle='--', alpha=0.7, label='θ=90° → eşit süperpozisyon')ax.axvline(x=180, color='#f59e0b', linestyle='--', alpha=0.7, label='θ=180° → tam çevirme (X kapısı)')ax.set_xlabel('θ (derece)', fontsize=12)ax.set_ylabel('Olasılık', fontsize=12)ax.set_title('Ry(θ) Kapısı — Açıya Göre Olasılık', fontsize=14, fontweight='bold')ax.legend(fontsize=11, loc='right')ax.grid(True, alpha=0.3)ax.set_xlim(0, 360)ax.set_ylim(-0.05, 1.05)plt.tight_layout()plt.show()print("📌 θ=0° → |0⟩ (değişmez)")print("📌 θ=90° → (|0⟩+|1⟩)/√2 (eşit süperpozisyon)")print("📌 θ=180° → |1⟩ (tam çevirme = X kapısı)")print("📌 θ=360° → |0⟩ (tam tur, başa döndü)")

## 5. Çok-Qubit Kapıları### CNOT (Controlled-NOT)En önemli 2-qubit kapısı. Dolanıklık oluşturur!- **Kontrol qubit** 0 ise → hedef qubit'e dokunma- **Kontrol qubit** 1 ise → hedef qubit'i çevir (X uygula)$$\text{CNOT} = \begin{pmatrix} 1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0 \end{pmatrix}$$| Giriş | Çıkış ||:---:|:---:|| ∣00⟩ | ∣00⟩ || ∣01⟩ | ∣01⟩ || ∣10⟩ | ∣11⟩ ← çevrildi! || ∣11⟩ | ∣10⟩ ← çevrildi! |### Diğer Önemli Kapılar| Kapı | Qubit | İşlev ||------|-------|-------|| CZ | 2 | Kontrol=1 ise hedefte Z uygula || SWAP | 2 | İki qubit'in durumunu değiştir || Toffoli (CCX) | 3 | 2 kontrol + 1 hedef (AND kapısı) |

In [ ]:
import pennylane as qmlimport numpy as npdev = qml.device('default.qubit', wires=3, shots=1000)# CNOT kapısı — tüm giriş-çıkış kombinasyonlarıprint("══════ CNOT Kapısı Doğruluk Tablosu ══════\n")for a in [0, 1]:    for b in [0, 1]:        @qml.qnode(dev)        def cnot_test(a, b):            if a: qml.PauliX(wires=0)            if b: qml.PauliX(wires=1)            qml.CNOT(wires=[0, 1])            return qml.probs(wires=[0, 1])                probs = cnot_test(a, b)        result = np.argmax(probs)        out_a, out_b = result // 2, result % 2        print(f"  |{a}{b}⟩  →  |{out_a}{out_b}⟩   {'← çevrildi!' if a==1 and out_b!=b else ''}")# Bell durumu oluşturma: H + CNOTprint("\n══════ H + CNOT = Bell Durumu ══════")@qml.qnode(dev)def bell():    qml.Hadamard(wires=0)    qml.CNOT(wires=[0, 1])    return qml.counts()print(f"\nDevre:\n{qml.draw(bell)()}")result = bell()print(f"\nSonuçlar (1000 shot): {dict(sorted(result.items()))}")print("→ Sadece |00⟩ ve |11⟩ çıkıyor = DOLANIKLIK!")

## 6. Kapı Özet Tablosu| Kapı | Simge | Matris | Etki | QML'de Kullanım ||------|-------|--------|------|-----------------|| X | ━[X]━ | [[0,1],[1,0]] | Bit çevirme | Durum hazırlama || Z | ━[Z]━ | [[1,0],[0,-1]] | Faz çevirme | Oracle (Grover) || H | ━[H]━ | 1/√2 [[1,1],[1,-1]] | Süperpozisyon | Başlangıç hazırlama || Ry(θ) | ━[Ry]━ | Rotasyon | θ açısıyla döndürme | **Variasyonel parametre** || CNOT | ━●━⊕━ | 4×4 matris | Koşullu çevirme | **Dolanıklık** |> **QML'de en çok kullanılanlar:** Ry(θ), Rz(θ) (parametreli), CNOT (dolanıklık)---## 📝 Özet| Kavram | Anahtar Bilgi ||--------|---------------|| Kuantum kapısı | Üniter matris → tersinebilir, bilgi kaybolmaz || Pauli X, Y, Z | 180° döndürmeler, temel yapı taşları || Hadamard (H) | Süperpozisyon oluşturur, kendi tersi (HH=I) || Ry(θ), Rz(θ) | QML'nin ayarlanabilir parametreleri (sinir ağı ağırlıkları gibi) || CNOT | Dolanıklık oluşturur, 2-qubit kontrollü kapı |---*Dr. Buket Toptaş — Kuantum Makine Öğrenmesi Dersi*